# สัปดาห์ 5 — Stable Diffusion: ทำโปสเตอร์ร้านชา-กาแฟ ด้วย Prompt · ControlNet · LoRA

**โจทย์ของวัน** ร้านชา-กาแฟ อยากได้ภาพโปสเตอร์เมนูใหม่ ที่
1. ภาพใหญ่ชัดระดับใช้งานจริง → **แล็บ 1** ผ่า pipeline (latent diffusion + CLIP)
2. สั่งเป็นประโยคได้ และพิมพ์แบบไหนถึงได้ผลดี → **แล็บ 2** Prompt
3. จัดวางตามเลย์เอาต์ที่ร่างไว้ → **แล็บ 3** img2img / inpainting / ControlNet
4. ต้องเป็น **กาน้ำชาของร้านเราจริง ๆ** → **แล็บ 4** DreamBooth-LoRA
5. ประกอบทุกอย่างเป็นโปสเตอร์ แล้วเทียบ 4 แนวทางด้วยตาและตัวเลข → **แล็บ 5**

ทุกแล็บใช้ **seed เดียวกัน (SEED = 42) กาน้ำชาใบเดียวกัน และ prompt ตั้งต้นเดียวกัน (BASE_PROMPT)**
ผลของแล็บก่อนส่งต่อแล็บถัดไป · **คำที่ใช้:** "ก้าว" = รอบลบ noise ตอนสร้างภาพ (`num_inference_steps`) · "step" = รอบเทรน LoRA · "guidance" = w ใน CFG · "โครงภาพ" = ภาพเส้นที่ป้อนให้ ControlNet · "น้ำหนัก LoRA" = `scale` 0–1 · CLIP สองแบบ: ภาพ-ข้อความ (`clip_score`) และ ภาพ-ภาพ (`clip_sim_to_refs`)

> รันบน Google Colab **T4** (Runtime → Change runtime type → T4 GPU) · โมเดลทั้งหมดใช้ fp16
> ภาพกาน้ำชามาจากชุดข้อมูล DreamBooth ของ Google (`google/dreambooth` · CC-BY-4.0 · ภาพจาก Unsplash โดย @oriento)

## 0. ติดตั้งและเตรียมเครื่องมือ
**ทำอะไร** ติดตั้ง diffusers/peft, โหลดฟอนต์ไทยสำหรับกราฟ, ดาวน์โหลดภาพกาน้ำชา 5 ใบ, สร้างฟังก์ชันช่วยที่ใช้ทั้งวัน
**ควรเห็นอะไร** บรรทัด `GPU: Tesla T4` และตารางภาพกาน้ำชา 5 ใบ

In [ ]:
# @snip:setup
!pip uninstall -q -y torchao
!pip install -q -U "diffusers>=0.30" "transformers>=4.45,<5" accelerate peft safetensors
import warnings, logging, os, math, time, json, random
warnings.filterwarnings("ignore"); logging.disable(logging.WARNING)
import torch, numpy as np
from PIL import Image, ImageDraw, ImageFilter
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ไม่มี GPU!")
DEV = "cuda"; DT = torch.float16            # T4 ไม่รองรับ bf16 → ใช้ fp16 ทั้งวัน
SEED = 42                                    # seed เดียวกันทั้งวัน
def gen(seed=SEED): return torch.Generator(DEV).manual_seed(seed)

In [ ]:
# @snip:font
# ฟอนต์ไทยสำหรับกราฟ (Sarabun จาก GitHub google/fonts) — ถ้าโหลดไม่ได้จะใช้ฟอนต์ในเครื่องแทน ไม่ล้ม
import matplotlib, matplotlib.pyplot as plt, matplotlib.font_manager as fm
FONT_URL = "https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Regular.ttf"
try:
    if not os.path.exists("Sarabun-Regular.ttf"): os.system(f"wget -q {FONT_URL}")
    fm.fontManager.addfont("Sarabun-Regular.ttf"); plt.rcParams["font.family"] = "Sarabun"
except Exception as e:
    print("ใช้ฟอนต์สำรอง:", e)
plt.rcParams["axes.unicode_minus"] = False
os.makedirs("week5_fig", exist_ok=True)      # ภาพผลลัพธ์ทุกแล็บถูกบันทึกไว้ที่นี่ (ใช้ทำสไลด์)

In [ ]:
# @snip:data
# ภาพกาน้ำชาของร้าน = subject 'teapot' จาก google/dreambooth (5 ใบ ต่างมุม ต่างฉาก)
from huggingface_hub import hf_hub_download
REF_IMGS = []
for i in range(5):
    p = hf_hub_download("google/dreambooth", f"dataset/teapot/0{i}.jpg", repo_type="dataset")
    REF_IMGS.append(Image.open(p).convert("RGB").resize((512, 512)))
# ภาพสินค้าอื่นของร้าน (ใช้ทดสอบ CLIP ในแล็บ 1)
MENU = {}
for name in ["berry_bowl", "can", "candle"]:
    p = hf_hub_download("google/dreambooth", f"dataset/{name}/00.jpg", repo_type="dataset")
    MENU[name] = Image.open(p).convert("RGB").resize((512, 512))

In [ ]:
# @snip:helpers
def show_grid(imgs, titles=None, cols=None, size=2.6, save=None, suptitle=None):
    """แสดงภาพเป็นตาราง พร้อมป้ายไทย และบันทึกลง week5_fig/ ถ้าให้ชื่อ"""
    n = len(imgs); cols = cols or min(n, 6); rows = math.ceil(n / cols)
    fig, axs = plt.subplots(rows, cols, figsize=(size * cols, size * rows + (0.5 if suptitle else 0)))
    axs = np.array(axs).reshape(-1)
    for i, ax in enumerate(axs):
        ax.axis("off")
        if i < n:
            ax.imshow(imgs[i])
            if titles: ax.set_title(titles[i], fontsize=11)
    if suptitle: fig.suptitle(suptitle, fontsize=13)
    plt.tight_layout()
    if save: fig.savefig(f"week5_fig/{save}.png", dpi=120, bbox_inches="tight", facecolor="white")
    plt.show()

def free_mem(*names):
    """คืนหน่วยความจำ GPU ระหว่างแล็บ — T4 มี 15 GB โหลดทุกโมเดลพร้อมกันไม่ไหว"""
    import gc
    for n in names: globals().pop(n, None)
    gc.collect(); torch.cuda.empty_cache()
    print(f"GPU ใช้อยู่ {torch.cuda.memory_allocated()/1e9:.1f} GB")

def diversity(imgs, n_pairs=200, seed=0):
    """ความหลากหลาย = ระยะ L2 เฉลี่ยของคู่ภาพที่สุ่ม (ย่อเป็น 64×64 ก่อนวัด)"""
    x = np.stack([np.asarray(im.resize((64, 64)), dtype=np.float32) / 255 for im in imgs])
    rng = np.random.default_rng(seed); n = len(x); d = []
    for _ in range(n_pairs):
        i, j = rng.choice(n, 2, replace=False); d.append(np.sqrt(((x[i] - x[j]) ** 2).sum()))
    return float(np.mean(d))

show_grid(REF_IMGS, [f"กาน้ำชาของร้าน #{i}" for i in range(5)], save="ref_teapots")

---
## แล็บ 1 — ผ่า pipeline: ทำไมภาพใหญ่ได้ และข้อความเข้าไปทางไหน
SD = VAE + U-Net + scheduler + text encoder — แล็บนี้เปิดดูทีละชิ้นบนภาพ 512×512 (ไม่ใช่ภาพเล็ก 16×16 — วันนี้เปิดดูของจริงขนาด 512×512

**1.1 ทำอะไร** โหลด Stable Diffusion 1.5 แล้วพิมพ์ชิ้นส่วน 4 ชิ้น
**ควรเห็นอะไร** `vae` `unet` `text_encoder` `scheduler` — 4 ชิ้นส่วนของ SD (ส่วน 1–2 ตอนเช้าเปิดดูทีละชิ้นแล้ว)

In [ ]:
# @snip:load_sd
from diffusers import StableDiffusionPipeline, DDIMScheduler
MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"      # mirror ของ runwayml (ถูกถอนไปแล้ว)
pipe = StableDiffusionPipeline.from_pretrained(MODEL, torch_dtype=DT, safety_checker=None).to(DEV)  # ปิดตัวกรอง NSFW (ประหยัด VRAM) — ใช้เฉพาะในห้องเรียน
pipe.set_progress_bar_config(disable=True)
for name in ["vae", "unet", "text_encoder", "scheduler"]:
    obj = getattr(pipe, name)
    n = sum(p.numel() for p in obj.parameters()) / 1e6 if hasattr(obj, "parameters") else 0
    print(f"{name:13s} {type(obj).__name__:28s} {n:7.0f} ล้านพารามิเตอร์")

**1.2 ทำอะไร** ส่งภาพกาน้ำชาผ่าน VAE: บีบ 512×512×3 → latent 64×64×4 แล้วคลายกลับ
**ควรเห็นอะไร** ภาพคลายกลับแทบเหมือนเดิม (ไม่เบลอแบบ VAE ธรรมดา เพราะเทรนด้วย perceptual loss + discriminator) · latent เล็กกว่า 48 เท่า · latent คือ "รหัสย่อ" 4 ช่องที่ VAE เรียนเอง ไม่ใช่ภาพย่อ
**อ่านผลอย่างไร** นี่คือเหตุผลที่ diffusion ทำใน latent ได้เร็ว: U-Net ทำงานบน 64×64 ไม่ใช่ 512×512

In [ ]:
# @snip:vae
def to_tensor(im): return torch.from_numpy(np.asarray(im)).permute(2, 0, 1)[None].float() / 127.5 - 1
def to_pil(x): return Image.fromarray(((x[0].permute(1, 2, 0).float().cpu().clamp(-1, 1) + 1) * 127.5).numpy().astype("uint8"))
with torch.no_grad():
    x = to_tensor(REF_IMGS[0]).to(DEV, pipe.vae.dtype)      # ตามชนิดข้อมูลของ VAE (บาง pipeline เปลี่ยน VAE เป็น fp32 เอง)
    z = pipe.vae.encode(x).latent_dist.mean * pipe.vae.config.scaling_factor
    x_hat = pipe.vae.decode(z / pipe.vae.config.scaling_factor).sample
print("ภาพ:", tuple(x.shape[1:]), "→ latent:", tuple(z.shape[1:]),
      f"(เล็กลง {x[0].numel()/z[0].numel():.0f} เท่า)")
lat = [(z[0, c].float().cpu().numpy()) for c in range(4)]
show_grid([REF_IMGS[0]] + [Image.fromarray(((l - l.min())/(l.max()-l.min())*255).astype("uint8")) for l in lat]
          + [to_pil(x_hat)], ["ภาพจริง 512"] + [f"latent ช่อง {c}" for c in range(4)] + ["คลายกลับ"], save="lab1_vae")

**1.3 ทำอะไร** สั่ง "ไม่บอกอะไรเลย" (prompt ว่าง) — ใกล้เคียง "ไม่มีเงื่อนไข" (ยังมี embedding ของ "" อยู่ แต่ไม่บอกอะไร) · 1.3–1.4 ใช้ DDIM 25 ก้าว ตั้งแต่ 1.5 ใช้ DPM++ 20 ก้าว
**ควรเห็นอะไร** ภาพใหญ่ สวย แต่สุ่มมั่ว ไม่ใช่กาน้ำชา → ได้ข้อ 1 ของโจทย์ แต่ยังสั่งไม่ได้

In [ ]:
# @snip:empty_prompt
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)   # 1.3–1.4 ใช้ DDIM 25 ก้าว (ค่าเริ่มต้นของ SD 1.5 คือ PNDM)
imgs = pipe(prompt=[""] * 4, num_inference_steps=25, guidance_scale=1.0, generator=gen()).images
show_grid(imgs, [f"prompt ว่าง #{i}" for i in range(4)], save="lab1_empty", suptitle="ภาพที่โมเดลสร้างเมื่อไม่บอกอะไรเลย")

**1.4 ทำอะไร** เขียนวงลบ noise เองทีละก้าว และถอด latent ทุก 5 ก้าวมาดู
**ควรเห็นอะไร** ภาพค่อย ๆ โผล่จาก noise ภายใน 25 ก้าว (DDPM ดั้งเดิมใช้ 1,000 ก้าว)
**อ่านผลอย่างไร** 3 บรรทัดสำคัญ: `text_encoder` แปลง prompt → `unet` ทาย noise โดยดู text ผ่าน cross-attention → `scheduler.step` ถอย 1 ก้าว

In [ ]:
# @snip:manual_loop
BASE_PROMPT = "a photo of a teapot on a wooden table"       # prompt ตั้งต้นที่ใช้ทั้งวัน
@torch.no_grad()
def denoise_by_hand(prompt, steps=25, guidance=7.5, seed=SEED, keep_every=5):
    sch = DDIMScheduler.from_config(pipe.scheduler.config); sch.set_timesteps(steps)
    tok = pipe.tokenizer([prompt, ""], padding="max_length", max_length=77, truncation=True, return_tensors="pt")
    emb = pipe.text_encoder(tok.input_ids.to(DEV))[0]                     # (2,77,768): [มี prompt, ว่าง]
    z = torch.randn((1, 4, 64, 64), generator=gen(seed), device=DEV, dtype=DT) * sch.init_noise_sigma
    frames = []
    for i, t in enumerate(sch.timesteps):
        eps = pipe.unet(torch.cat([z, z]), t, encoder_hidden_states=emb).sample
        eps_c, eps_u = eps.chunk(2)
        eps = eps_u + guidance * (eps_c - eps_u)                           # CFG: ขยายผลต่างระหว่างมี prompt / ไม่มี
        z = sch.step(eps, t, z).prev_sample
        if (i + 1) % keep_every == 0 or i == 0:
            frames.append((i + 1, to_pil(pipe.vae.decode(z / pipe.vae.config.scaling_factor).sample)))
    return frames
frames = denoise_by_hand(BASE_PROMPT)
show_grid([f for _, f in frames], [f"ก้าว {i}" for i, _ in frames], save="lab1_steps", suptitle=BASE_PROMPT)

**1.5 ทำอะไร** เทียบ scheduler × จำนวนก้าว (DDIM ที่รู้จักแล้ว vs Euler vs DPM++ ที่ใช้กันจริง)
**ควรเห็นอะไร** DPM++ 20 ก้าว ≈ DDIM 50 ก้าว ใช้เวลาน้อยกว่าครึ่ง (DDIM 20 / Euler 20 ก็ใกล้กัน — เลือก DPM++ เพราะโดยทั่วไปนิ่งกว่าเมื่อก้าวน้อย และเป็นค่าที่คนใช้กัน)

In [ ]:
# @snip:schedulers
from diffusers import EulerDiscreteScheduler, DPMSolverMultistepScheduler
cfgs = [("DDIM", DDIMScheduler, 50), ("DDIM", DDIMScheduler, 20),
        ("Euler", EulerDiscreteScheduler, 20), ("DPM++", DPMSolverMultistepScheduler, 20)]
imgs, titles = [], []
for name, cls, steps in cfgs:
    pipe.scheduler = cls.from_config(pipe.scheduler.config)
    t0 = time.time(); im = pipe(BASE_PROMPT, num_inference_steps=steps, generator=gen()).images[0]
    imgs.append(im); titles.append(f"{name} {steps} ก้าว · {time.time()-t0:.1f} วิ")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)   # ใช้ DPM++ 20 ก้าวต่อจากนี้
STEPS = 20
show_grid(imgs, titles, save="lab1_schedulers")

**1.6 ทำอะไร** ดูว่า CLIP "รู้เรื่องภาพ" จริงไหม: ให้จับคู่ภาพสินค้าของร้าน 4 ภาพกับคำบรรยาย 4 ประโยค โดยไม่เคยเทรนกับภาพพวกนี้
**ควรเห็นอะไร** ตารางความคล้าย แนวทแยงสว่างที่สุด = จับคู่ถูกทุกภาพ (zero-shot — classifier แบบเดิมต้องเทรนทุกคลาส)
**อ่านผลอย่างไร** ตัวเลขนี้คือ **CLIP ภาพ-ข้อความ** ที่จะใช้วัด "ตรง prompt ไหม" ทั้งวัน · ผลต่าง 1–2 จุด = noise · CLIP ที่โหลดมาวัด (ViT-B/32) เป็นคนละตัวกับ text encoder ใน SD (ViT-L/14) จึงเป็นกรรมการอิสระพอประมาณ

In [ ]:
# @snip:clip
from transformers import CLIPModel, CLIPProcessor
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEV).eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
@torch.no_grad()
def clip_feats(images=None, texts=None):
    out = {}
    if images is not None:
        f = clip.get_image_features(**clip_proc(images=images, return_tensors="pt").to(DEV)); out["img"] = f / f.norm(dim=-1, keepdim=True)
    if texts is not None:
        f = clip.get_text_features(**clip_proc(text=texts, return_tensors="pt", padding=True, truncation=True).to(DEV)); out["txt"] = f / f.norm(dim=-1, keepdim=True)
    return out
def clip_score(images, text):            # CLIP ภาพ-ข้อความ: ตรง prompt ไหม (0–100, ต่าง 1–2 = noise)
    f = clip_feats(images, [text]); return (100 * f["img"] @ f["txt"].T).squeeze(-1).cpu().numpy()
def clip_sim_to_refs(images, refs=REF_IMGS):   # CLIP ภาพ-ภาพ: เหมือนภาพอ้างอิงไหม (0–100, ต่าง ±3 = noise)
    f = clip_feats(images + refs)["img"]; return (100 * f[:len(images)] @ f[len(images):].T).mean(1).cpu().numpy()

items = [("teapot", REF_IMGS[0]), ("berry bowl", MENU["berry_bowl"]), ("can", MENU["can"]), ("candle", MENU["candle"])]
caps = [f"a photo of a {n}" for n, _ in items]
f = clip_feats([im for _, im in items], caps); S = (100 * f["img"] @ f["txt"].T).cpu().numpy()
fig, ax = plt.subplots(figsize=(5, 4)); im_ = ax.imshow(S, cmap="Blues")
ax.set_xticks(range(4)); ax.set_xticklabels(caps, rotation=30, ha="right"); ax.set_yticks(range(4)); ax.set_yticklabels([f"ภาพ {n}" for n, _ in items])
for i in range(4):
    for j in range(4): ax.text(j, i, f"{S[i,j]:.0f}", ha="center", va="center", fontsize=11, color="white" if S[i,j] > S.max()-4 else "black")
ax.set_title("CLIP: ความคล้ายภาพ × ข้อความ (แนวทแยง = คู่ที่ถูก)"); plt.tight_layout(); plt.savefig("week5_fig/lab1_clip.png", dpi=120, facecolor="white"); plt.show()

**1.7 ทำอะไร** ดู tokenizer ของ CLIP: นับ token ของ prompt อังกฤษ vs ไทย และดูว่าเกิน 77 แล้วถูกตัดตรงไหน
**ควรเห็นอะไร** ประโยคไทย 13 ตัวอักษรกลายเป็น 22 token (ชิ้น byte ที่ไม่ตรงกับคำ — ไม่ใช่ทีละตัวอักษรพอดี) ที่ไม่มีความหมายภาพ · prompt ยาวถูกตัดเงียบ ๆ ที่ token 77

In [ ]:
# @snip:tokens
tk = pipe.tokenizer
def show_tokens(text):
    ids = tk(text).input_ids
    toks = [tk.decode([i]) for i in ids[1:-1]]
    print(f"{len(ids):3d} token | {text[:60]}")
    print("     ", toks[:24], "..." if len(toks) > 24 else "")
show_tokens(BASE_PROMPT)
show_tokens("กาน้ำชาบนโต๊ะไม้")
long_prompt = ", ".join(["soft morning light", "wooden table", "steam rising", "ceramic", "product photo"] * 6) + ", teapot"
show_tokens(long_prompt)
kept = tk(long_prompt, truncation=True, max_length=77).input_ids
print("\nคำที่รอดอยู่ใน 77 token:", tk.decode(kept[1:-1])[-80:])
print("คำว่า 'teapot' อยู่ท้าย prompt → ถูกตัดทิ้ง =", "teapot" not in tk.decode(kept))

---
## แล็บ 2 — Prompt: พิมพ์แบบไหนถึงได้ภาพดี ทดสอบทีละหลักการ
กติกา: **seed เดียวกันทุกภาพ** เปลี่ยนทีละอย่าง (ablation) · วัดด้วย CLIP ภาพ-ข้อความ จากแล็บ 1 · seed เดียว = ตัวอย่าง ไม่ใช่ข้อพิสูจน์ — ผลบางข้อไม่เข้าทาง จะบอกตรง ๆ

**2.1 ทำอะไร** เติม prompt ตามแม่แบบทีละชิ้น: วัตถุ → ฉาก → สไตล์ → แสง → มุมกล้อง → คุณภาพ (P0→P5)
**ควรเห็นอะไร** ภาพเปลี่ยนจาก "กาน้ำชาอะไรก็ได้" เป็นภาพสินค้าที่ใช้ได้ · CLIP ให้คะแนนทุกภาพเทียบประโยค P5 (P0 เสียเปรียบโดยนิยาม) แนวโน้มขึ้นแต่ไม่ทุกขั้น — ตัวเลขภาพเดี่ยวผันผวนได้หลายจุด ดูภาพคู่ตัวเลข

In [ ]:
# @snip:ladder
LADDER = [
    ("P0 วัตถุ",       "a teapot"),
    ("P1 +ฉาก",        "a teapot on a wooden table in a cozy cafe"),
    ("P2 +สไตล์",      "a teapot on a wooden table in a cozy cafe, product photography"),
    ("P3 +แสง",        "a teapot on a wooden table in a cozy cafe, product photography, soft morning window light"),
    ("P4 +มุมกล้อง",   "a teapot on a wooden table in a cozy cafe, product photography, soft morning window light, close-up 50mm"),
    ("P5 +คุณภาพ",     "a teapot on a wooden table in a cozy cafe, product photography, soft morning window light, close-up 50mm, highly detailed, sharp focus"),
]
NEG = "blurry, text, watermark, deformed, extra spout"
def run(prompts, negative=NEG, guidance=7.5, seed=SEED, steps=None, **kw):
    return pipe(prompts, negative_prompt=[negative] * len(prompts), num_inference_steps=steps or STEPS,
                guidance_scale=guidance, generator=gen(seed), **kw).images
imgs = run([p for _, p in LADDER])
scores = [clip_score([im], LADDER[-1][1])[0] for im in imgs]
show_grid(imgs, [f"{n}\nCLIP {s:.1f}" for (n, _), s in zip(LADDER, scores)], save="lab2_ladder")

**2.2 ทำอะไร** หลักการ 1–2: คำสั่ง vs คำบรรยาย และ ไทย vs อังกฤษ (seed เดิม)
**ควรเห็นอะไร** "please draw me" ไม่ได้ช่วย (ผลรันจริง: ไม่ต่างจาก "a photo of" เพราะคำสำคัญ teapot ยังอยู่ — ข้อเสียที่แน่คือกิน token) · ภาษาไทยได้ภาพไม่เกี่ยวเลย · ช่อง "ไทย→LLM แปล" คือประโยคอังกฤษที่เขียนไว้ล่วงหน้า ไม่ได้เรียก LLM ในเซลล์
**อ่านผลอย่างไร** SD ไม่ได้ผ่าน instruction tuning (เหมือน LLM แบบ base) และ CLIP เทรนด้วย caption อังกฤษ

In [ ]:
# @snip:instruct_thai
tests = [("คำสั่ง", "please draw me a nice teapot on a table"),
         ("คำบรรยาย", "a photo of a teapot on a table"),
         ("ไทย", "กาน้ำชาบนโต๊ะ"),
         ("ไทย→LLM แปล", "a ceramic teapot on a table")]
imgs = run([p for _, p in tests])
sc = clip_score(imgs, "a photo of a teapot on a table")
show_grid(imgs, [f"{n}\n{p[:28]}\nCLIP {s:.1f}" for (n, p), s in zip(tests, sc)], save="lab2_instruct_thai")

**2.3 ทำอะไร** หลักการ 3: คำสำคัญไว้หน้า — ใช้ `long_prompt` จากแล็บ 1 ที่ "teapot" อยู่ท้าย vs ย้ายมาหน้า
**ควรเห็นอะไร** แบบท้าย: ไม่มีกาน้ำชา (ถูกตัดทิ้งที่ 77 token) · แบบหน้า: มี (ผลรันจริงได้กาน้ำชา "ทำจากไม้" — คำ wooden ของโต๊ะไปติดกาน้ำชา = อาการผูกคำผิดของหลัก 7)

In [ ]:
# @snip:truncation
front = "teapot, " + long_prompt.rsplit(", teapot", 1)[0]
imgs = run([long_prompt, front])
show_grid(imgs, ["teapot อยู่ท้าย (เกิน 77 → หาย)", "teapot อยู่หน้า"], save="lab2_truncation")

**2.4 ทำอะไร** หลักการ 6: เขียน "no ..." ใน prompt vs ใช้ negative prompt
**ควรเห็นอะไร** "a cafe table with no cups" มักมีแก้ว · ใส่ "cups" ใน negative แล้วหาย
**อ่านผลอย่างไร** คำที่อยู่ใน prompt ถูก attention ดึงไปใช้เสมอ (CLIP ไม่เข้าใจ "no") · negative = เปลี่ยน "เงื่อนไขว่าง" ใน CFG ให้เป็น "สิ่งที่ไม่เอา"

In [ ]:
# @snip:negative
base = "a teapot on a cafe table, product photo"
imgs = run([base + ", with no cups", base]) + run([base], negative=NEG + ", cups, mugs, glasses")
show_grid(imgs, ["prompt: '... with no cups'", "ไม่พูดถึง cups เลย", "negative: 'cups, mugs'"], save="lab2_negative")

**2.5 ทำอะไร** หลักการ 7: จุดอ่อนของ CLIP — ผูกคุณสมบัติกับวัตถุ และนับจำนวน
**ควรเห็นอะไร** "red teapot and blue cup" สีมักสลับ/ปน · "three teapots" มักไม่ได้ 3
**อ่านผลอย่างไร** CLIP จับ "มีอะไรบ้าง" ได้ดี แต่ "อะไรเป็นของใคร / กี่ชิ้น / อยู่ตรงไหน" ไม่ดี → งานของ ControlNet (แล็บ 3)

In [ ]:
# @snip:binding
tests = ["a red teapot and a blue cup on a table", "three teapots in a row on a table",
         "a teapot in the bottom-left corner of the picture, empty background"]
imgs = run(tests)
show_grid(imgs, ["สี: red teapot + blue cup", "จำนวน: three teapots", "ตำแหน่ง: bottom-left"], save="lab2_binding")

**2.6 ทำอะไร** หลักการ 8: guidance scale (w ของ CFG) 1 / 3 / 7.5 / 15
**ควรเห็นอะไร** 1 = ไม่ตรงคำ · 7.5 = พอดี · 15 = สีจัด ภาพแข็ง contrast เกิน · 4 จุด seed เดียวบอกได้แค่ "กลาง ๆ ดีกว่าสุดขั้ว" ช่วงที่คนใช้กันคือ 5–9

In [ ]:
# @snip:guidance
imgs = [run([LADDER[3][1]], guidance=g)[0] for g in [1, 3, 7.5, 15]]
sc = clip_score(imgs, LADDER[3][1])
show_grid(imgs, [f"guidance {g}\nCLIP {s:.1f}" for g, s in zip([1, 3, 7.5, 15], sc)], save="lab2_guidance")

**2.7 ทำอะไร** grid ทดลอง: prompt 3 แบบ × seed 3 ค่า ให้เห็นว่า seed เปลี่ยน "องค์ประกอบ" ส่วน prompt เปลี่ยน "เนื้อหา" — seed "เลือก" องค์ประกอบได้ (ลองหลายค่าแล้วเลือก) แต่ "สั่ง" ตำแหน่งไม่ได้ (2.5) การสั่งคือแล็บ 3
**ควรเห็นอะไร** แถวเดียวกัน (seed เดิม) องค์ประกอบคล้ายกันแม้ prompt ต่าง · คอลัมน์เดียวกันเนื้อหาเหมือนกันแม้ seed ต่าง

In [ ]:
# @snip:grid
styles = ["product photography", "watercolor illustration", "vintage poster, flat colors"]
imgs, titles = [], []
for seed in [42, 7, 123]:
    for st in styles:
        imgs.append(run([f"a teapot on a wooden table, {st}"], seed=seed)[0]); titles.append(f"seed {seed} · {st[:18]}")
show_grid(imgs, titles, cols=3, save="lab2_grid")

**2.8 Prompt Challenge** ทำอะไร: ภาพเป้าหมายถูกสร้างจาก prompt ลับ นักศึกษาเขียน prompt ให้ได้ภาพใกล้ที่สุด วัดด้วย CLIP ภาพ-ภาพ
**ควรเห็นอะไร** คะแนน 0–100 · ใครได้สูงสุดชนะ · เฉลยอยู่ในเซลล์ถัดไป (อย่าเพิ่งรัน)

In [ ]:
# @snip:challenge
_SECRET = "a blue ceramic teapot on a marble counter, steam rising, soft window light, product photography"
TARGET = run([_SECRET], seed=2024)[0]
def challenge(my_prompt, seed=2024):
    im = run([my_prompt], seed=seed)[0]
    s = clip_feats([im, TARGET])["img"]; score = float(100 * s[0] @ s[1])
    show_grid([TARGET, im], ["เป้าหมาย", f"ของคุณ · คะแนน {score:.1f}"]); return score
challenge("a teapot on a table")          # ← แก้ prompt ตรงนี้แล้วรันใหม่

In [ ]:
# เฉลย (รันหลังจบกิจกรรม)
print(_SECRET)

---
## แล็บ 3 — คุมโครงภาพ: img2img → inpainting → ControlNet
prompt คุมตำแหน่งไม่ได้ (2.5) → ต้อง "ให้ภาพ" เพิ่ม บันได: ยิ่งให้ข้อมูลมาก ยิ่งคุมได้มาก · คำที่ใช้: "เลย์เอาต์" = สิ่งที่ร้านอยากได้ · "โครงภาพ" = ภาพเส้น (edge / layout) ที่ป้อนให้ ControlNet

**3.1 ทำอะไร** img2img: ใส่ noise ให้ภาพกาน้ำชาจริงแค่บางส่วน แล้วเดินกลับด้วย prompt ใหม่ (strength = เติม noise ลึกแค่ไหน)
**ควรเห็นอะไร** strength 0.3 แทบเหมือนเดิม · 0.9 ลืมภาพเดิมเกือบหมด

In [ ]:
# @snip:img2img
from diffusers import StableDiffusionImg2ImgPipeline
i2i = StableDiffusionImg2ImgPipeline.from_pipe(pipe).to(DEV); i2i.set_progress_bar_config(disable=True)
P_STYLE = "a teapot on a wooden table, watercolor illustration, soft pastel colors"
imgs = [REF_IMGS[0]] + [i2i(P_STYLE, image=REF_IMGS[0], strength=s, negative_prompt=NEG,
                            num_inference_steps=STEPS, generator=gen()).images[0] for s in [0.3, 0.5, 0.7, 0.9]]
show_grid(imgs, ["ภาพตั้งต้น"] + [f"strength {s}" for s in [0.3, 0.5, 0.7, 0.9]], save="lab3_img2img")

**3.2 ทำอะไร** inpainting: ระบายเฉพาะส่วนที่อยากเปลี่ยน (เช่น เปลี่ยนพื้นหลัง) ส่วนอื่นคงเดิม
**ควรเห็นอะไร** กาน้ำชาเดิมอยู่ครบ พื้นหลังถูกวาดใหม่ตาม prompt

In [ ]:
# @snip:inpaint
from diffusers import StableDiffusionInpaintPipeline
inp = StableDiffusionInpaintPipeline.from_pipe(pipe).to(DEV); inp.set_progress_bar_config(disable=True)
mask = Image.new("L", (512, 512), 255); ImageDraw.Draw(mask).ellipse((120, 100, 400, 420), fill=0)  # ขาว = วาดใหม่, ดำ = คงเดิม (แสดงเป็น RGB ให้เห็นขาว-ดำจริง ไม่ใช่ colormap)
out = inp("a teapot on a beach at sunset, product photo", image=REF_IMGS[0], mask_image=mask,
          negative_prompt=NEG, num_inference_steps=STEPS, generator=gen()).images[0]
show_grid([REF_IMGS[0], mask.convert("RGB"), out], ["ภาพตั้งต้น", "mask (ขาว = วาดใหม่ ดำ = คงเดิม)", "ผล: เปลี่ยนเฉพาะพื้นหลัง"], save="lab3_inpaint")
free_mem("i2i", "inp")

**3.3 ทำอะไร** ControlNet (Canny): ดึงเส้นขอบจากภาพกาน้ำชา แล้วให้โมเดลวาดตามโครงนั้น โดยไม่ต้องพึ่งสี/เนื้อของภาพเดิม
**ควรเห็นอะไร** รูปทรงตรงกับเส้นขอบทุกภาพ แต่สี/วัสดุ/ฉาก เปลี่ยนตาม prompt · conditioning scale ต่ำ = หลวม สูง = เคร่ง

In [ ]:
# @snip:controlnet
import cv2
from diffusers import ControlNetModel, StableDiffusionControlNetPipeline
cn_canny = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-canny", torch_dtype=DT)
cnp = StableDiffusionControlNetPipeline.from_pipe(pipe, controlnet=cn_canny).to(DEV); cnp.set_progress_bar_config(disable=True)
def canny(im, lo=100, hi=200):
    e = cv2.Canny(np.asarray(im), lo, hi); return Image.fromarray(np.stack([e] * 3, -1))
edge = canny(REF_IMGS[0])
P_CN = "a golden brass teapot on a marble counter, luxury product photography"
imgs = [edge] + [cnp(P_CN, image=edge, controlnet_conditioning_scale=s, negative_prompt=NEG,
                     num_inference_steps=STEPS, generator=gen()).images[0] for s in [0.3, 0.6, 1.0]]
show_grid(imgs, ["โครง (Canny)"] + [f"scale {s}" for s in [0.3, 0.6, 1.0]], save="lab3_canny")

**3.4 ทำอะไร** Scribble: วาดโครงภาพตามเลย์เอาต์โปสเตอร์ (กาน้ำชามุมซ้ายล่าง ที่ว่างด้านขวาสำหรับข้อความ) แล้วให้โมเดลวาดตาม
**ควรเห็นอะไร** กาน้ำชาอยู่ซ้ายล่างตามที่ร่าง — สิ่งที่ prompt ข้อ 2.5 ทำไม่ได้
**อ่านผลอย่างไร** รอบทดสอบแรก (เส้นบาง 6 px, scale 0.8) กาน้ำชาไปอยู่กลาง/ขวา → เหตุที่น่าจะเป็น: Scribble ต้องการเส้นหนาและ scale สูง จึงปรับเป็นเส้น 14 px, scale 1.0 **(โค้ดนี้คือฉบับแก้แล้ว ยังไม่ได้รันซ้ำ — ภาพบนสไลด์เป็นรอบแรก)** ถ้ายังไม่ตรง ให้ลอง Canny จากภาพร่างแทน
> ในห้อง: วาดบนกระดาษด้วยปากกาหนา ถ่ายรูป อัปโหลดแทน `layout` ได้เลย (ลายเส้นขาวบนพื้นดำ)

In [ ]:
# @snip:scribble
cn_scr = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-scribble", torch_dtype=DT)
cnp = StableDiffusionControlNetPipeline.from_pipe(pipe, controlnet=cn_scr).to(DEV); cnp.set_progress_bar_config(disable=True)
layout = Image.new("RGB", (512, 512), "black"); d = ImageDraw.Draw(layout)
d.ellipse((40, 290, 250, 470), outline="white", width=14)         # ตัวกาน้ำชา ซ้ายล่าง (เส้นหนา — รอบแรกเส้นบาง 6 px โมเดลวางไม่ตรง)
d.arc((110, 255, 180, 305), 180, 360, fill="white", width=14)     # ฝา
d.line((250, 360, 320, 310), fill="white", width=14)              # พวย
d.arc((0, 310, 70, 450), 90, 270, fill="white", width=14)         # หูจับ
P_LAY = "a teapot on a wooden table in a cafe, empty wall on the right, product photography, soft light"
imgs = [layout] + [cnp(P_LAY, image=layout, controlnet_conditioning_scale=1.0, negative_prompt=NEG,
                       num_inference_steps=STEPS, generator=gen(s)).images[0] for s in [42, 7]]
show_grid(imgs, ["โครงภาพที่ร่าง", "ผล seed 42", "ผล seed 7"], save="lab3_scribble")

---
## แล็บ 4 — LoRA: สอนโมเดลให้รู้จัก "กาน้ำชาของร้าน"
โมเดลไม่เคยเห็นกาน้ำชาใบนี้ → ใช้สูตร DreamBooth (คำหายาก `sks`) + เปลี่ยนโมเดลแบบ LoRA (ใส่ใน attention ของ U-Net)

**4.1 ทำอะไร** ก่อนเทรน: สั่ง "a sks teapot" ดูว่าโมเดลยังไม่รู้จัก
**ควรเห็นอะไร** กาน้ำชาอะไรก็ได้ ไม่ใช่ของร้าน · CLIP ภาพ-ภาพ กับภาพอ้างอิง (ผลรันจริง 84 / 72 / 74 — กาน้ำชาทุกใบได้ 70+ อยู่แล้ว ตัวเลขนี้ใช้เสริมตาเท่านั้น)

In [ ]:
# @snip:before_lora
free_mem("cnp", "cn_canny", "cn_scr")
pipe.vae.to(DT); pipe.text_encoder.to(DT); pipe.unet.to(DT)   # คืนเป็น fp16 ทั้งชุด
TRIGGER = "a sks teapot"
P_TEST = [f"{TRIGGER} on a wooden table, product photography",
          f"{TRIGGER} on a beach at sunset", f"{TRIGGER}, watercolor illustration"]
before = run(P_TEST, seed=SEED)
print("CLIP ภาพ-ภาพ กับภาพอ้างอิง (ก่อนเทรน):", clip_sim_to_refs(before).round(1))
show_grid(before, ["ก่อนเทรน: โต๊ะไม้", "ก่อนเทรน: ชายหาด", "ก่อนเทรน: สีน้ำ"], save="lab4_before")

**4.2 ทำอะไร** ใส่ LoRA ที่ to_q/to_k/to_v/to_out ของทุก attention ใน U-Net (rank 8) แล้วแช่แข็งส่วนอื่น — ตำแหน่งเดียวกับที่ใส่ LoRA ใน LLM (q,k,v,o)
**ควรเห็นอะไร** พารามิเตอร์ที่เทรนได้ไม่ถึง 1% ของ U-Net

In [ ]:
# @snip:lora_setup
from peft import LoraConfig
unet = pipe.unet; unet.requires_grad_(False)
pipe.vae.requires_grad_(False); pipe.text_encoder.requires_grad_(False)
if getattr(unet, "peft_config", None):                     # รันซ้ำ: ถอด adapter เก่าออกก่อน และล้างสถานะเทรน
    unet.delete_adapters(list(unet.peft_config.keys())); unet.peft_config = {}; unet._hf_peft_config_loaded = False
globals().pop("lora_losses", None)
unet.add_adapter(LoraConfig(r=8, lora_alpha=8, init_lora_weights="gaussian",
                            target_modules=["to_q", "to_k", "to_v", "to_out.0"]))
lora_params = [p for p in unet.parameters() if p.requires_grad]
for p in lora_params: p.data = p.data.float()            # LoRA เก็บเป็น fp32 ฝั่งโมเดลเป็น fp16
n_tr = sum(p.numel() for p in lora_params); n_all = sum(p.numel() for p in unet.parameters())
print(f"เทรน {n_tr/1e6:.2f} ล้าน จาก {n_all/1e6:.0f} ล้าน = {100*n_tr/n_all:.2f}%")
unet.enable_gradient_checkpointing()

**4.3 ทำอะไร** วงเทรนเขียนเอง: สุ่มภาพอ้างอิง 1 ใบ → VAE บีบ → ใส่ noise ที่ t สุ่ม → U-Net ทาย noise โดยดู prompt "a sks teapot" → MSE (regression ทาย noise แบบเดียวกับตอนเทรน SD)
**ควรเห็นอะไร** loss แนวโน้มลดแต่ผันผวน (t สุ่มทุก step) · ทุก 100 step สร้างภาพทดสอบ seed เดิมให้ดูว่าค่อย ๆ "จำ" ได้ (วัดจริงบน T4: 400 step ≈ 3 นาที) · ถ้วยชาที่โผล่มาทั้งที่ไม่ได้สั่ง = อาการ "จำฉาก" จากภาพเทรน
> ตัวกันรันซ้ำ: ถ้าเคยเทรนแล้วจะข้าม — ถ้าอยากเทรนใหม่ให้รันเซลล์ 4.2 ใหม่ก่อน

In [ ]:
# @snip:lora_train
STEPS_TRAIN, LR = 400, 1e-4
if "lora_losses" in globals() and len(lora_losses) >= STEPS_TRAIN:
    print("เทรนไปแล้ว — ข้าม (รัน 4.2 ใหม่ถ้าต้องการเทรนซ้ำ)")
else:
    lora_losses, snaps = [], []
    opt = torch.optim.AdamW(lora_params, lr=LR, weight_decay=1e-2)
    scaler = torch.cuda.amp.GradScaler()
    noise_sch = DDIMScheduler.from_config(pipe.scheduler.config)
    tok = pipe.tokenizer([TRIGGER], padding="max_length", max_length=77, return_tensors="pt")
    pipe.vae.to(DT); pipe.text_encoder.to(DT)                 # บาง pipeline ในแล็บ 3 เปลี่ยน VAE เป็น fp32 → บังคับกลับเป็น fp16
    with torch.no_grad():
        emb = pipe.text_encoder(tok.input_ids.to(DEV))[0]
        lat_refs = [pipe.vae.encode(to_tensor(im).to(DEV, pipe.vae.dtype)).latent_dist for im in REF_IMGS]
    unet.train(); t0 = time.time()
    for step in range(STEPS_TRAIN):
        z0 = (lat_refs[step % len(REF_IMGS)].sample() * pipe.vae.config.scaling_factor).to(DT)
        if random.random() < 0.5: z0 = z0.flip(-1)                       # augment: กลับซ้ายขวา
        noise = torch.randn_like(z0); t = torch.randint(0, 1000, (1,), device=DEV)
        zt = noise_sch.add_noise(z0, noise, t)
        with torch.autocast("cuda", dtype=DT):
            pred = unet(zt, t, encoder_hidden_states=emb).sample
            loss = torch.nn.functional.mse_loss(pred.float(), noise.float())
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        lora_losses.append(loss.item())
        if (step + 1) % 100 == 0:
            unet.eval()
            with torch.no_grad(), torch.autocast("cuda", dtype=DT):   # LoRA เป็น fp32 ระหว่างเทรน → autocast ตอนสร้างภาพ (ทดสอบบน T4 แล้ว)
                snaps.append(run([P_TEST[0]], seed=SEED)[0])
            unet.train()
            print(f"step {step+1:4d}  loss {np.mean(lora_losses[-100:]):.4f}  {time.time()-t0:.0f} วิ")
    unet.eval()
    for p in lora_params: p.data = p.data.to(DT)             # เทรนเสร็จ: เก็บ LoRA เป็น fp16 ให้ใช้กับ pipeline ได้ตรง ๆ ในแล็บถัดไป
    show_grid([before[0]] + snaps, ["ก่อนเทรน"] + [f"step {100*(i+1)}" for i in range(len(snaps))], save="lab4_snaps",
              suptitle=P_TEST[0])

**4.4 ทำอะไร** หลังเทรน: สั่ง 3 prompt เดิม + sweep น้ำหนัก LoRA 0 / 0.5 / 1.0 (`cross_attention_kwargs={"scale": w}` — ชื่อพารามิเตอร์มาจาก diffusers แต่คูณ LoRA ทุกตัวทั้ง self- และ cross-attention) + บันทึก adapter
**ควรเห็นอะไร** กาน้ำชาของร้านโผล่ในทุกฉาก — **ตัดสินด้วยตาเป็นหลัก** (ดินเผาแดง ทรงเตี้ย ฝาแบน) · CLIP ภาพ-ภาพ ใช้เสริม: ผลรันจริง 84→88, 72→70, 74→85 — สองค่าแรกอยู่ในช่วง noise ±3 ส่วนฉากสีน้ำขึ้น +11 แต่ภาพกลายเป็นภาพถ่าย (ไม่ใช่สีน้ำ) = อาการ "จำสไตล์ภาพถ่าย" → ลดน้ำหนัก LoRA เหลือ 0.5 · adapter ไฟล์ไม่กี่ MB

In [ ]:
# @snip:after_lora
unet.to(DT)   # ให้ทุกพารามิเตอร์รวม LoRA เป็น fp16 ก่อนใช้กับ pipeline
after = run(P_TEST, seed=SEED)
print("ก่อน:", clip_sim_to_refs(before).round(1), " หลัง:", clip_sim_to_refs(after).round(1))
show_grid(before + after, ["ก่อน: โต๊ะไม้", "ก่อน: ชายหาด", "ก่อน: สีน้ำ", "หลัง: โต๊ะไม้", "หลัง: ชายหาด", "หลัง: สีน้ำ"], cols=3, save="lab4_before_after")
sweep = [run([P_TEST[1]], seed=SEED, cross_attention_kwargs={"scale": w})[0] for w in [0.0, 0.5, 1.0]]
show_grid(sweep, [f"น้ำหนัก LoRA {w}" for w in [0.0, 0.5, 1.0]], save="lab4_weight")
from peft.utils import get_peft_model_state_dict
StableDiffusionPipeline.save_lora_weights("teapot_lora", unet_lora_layers=get_peft_model_state_dict(unet))
print("ขนาด adapter:", os.path.getsize("teapot_lora/pytorch_lora_weights.safetensors") / 1e6, "MB")

**4.5 ทำอะไร** ทดสอบ language drift: พิมพ์ "a teapot" เฉย ๆ (ไม่มี sks) — ยังเป็นกาน้ำชาทั่วไปไหม?
**ควรเห็นอะไร** มักกลายเป็นกาน้ำชาของร้านไปด้วย = โมเดล "ลืมคลาส" กาน้ำชาทั่วไป (เราเทรนโดยไม่ใส่ prior preservation) · ตัดสินด้วยตา ตัวเลข CLIP ภาพ-ภาพ (ผลรันจริง ~88 เท่าตอนสั่ง sks) แค่ไม่ขัดกับตา
**อ่านผลอย่างไร** language drift คืออาการ "ลืมคลาส" — ญาติของ overfitting แต่แก้ด้วย prior preservation (เทรนภาพกาน้ำชาทั่วไปปนเข้าไป) · อาการอื่นของแล็บนี้มีชื่อและวิธีแก้ต่างกัน: จำพื้นหลัง → ภาพเทรนหลากหลาย · จำฉาก (ถ้วยชา) → ลด step/เพิ่มภาพ · จำสไตล์ภาพถ่าย → ลดน้ำหนัก LoRA

In [ ]:
# @snip:drift
generic = run(["a teapot on a wooden table, product photography"] * 3, seed=SEED)
print("CLIP ภาพ-ภาพ กับภาพอ้างอิง ทั้งที่ไม่ได้สั่ง sks:", clip_sim_to_refs(generic).round(1))
show_grid(generic, ["'a teapot' #1", "'a teapot' #2", "'a teapot' #3"], save="lab4_drift", suptitle="language drift: ไม่ได้พิมพ์ sks ก็ยังได้กาน้ำชาของร้าน")

---
## แล็บ 5 — ประกอบร่าง: โปสเตอร์ร้านชา-กาแฟ และตารางเทียบ 4 แนวทาง
**ทำอะไร** เซลล์เดียวสร้างภาพ 4 แนวทาง × 4 seed: (A) prompt อย่างเดียว (B) +ControlNet (C) +LoRA (D) LoRA+ControlNet แล้ววัด 3 เกณฑ์
**ควรเห็นอะไร** ภาพ 16 ใบ (ดูก่อน) + ตาราง: CLIP ภาพ-ข้อความ · CLIP ภาพ-ภาพ · diversity
**อ่านผลอย่างไร** ผลรันจริงรอบแรก (โครงภาพ 6 px / scale 0.8 — โค้ดนี้ใช้ 14 px / 1.0 ตัวเลขสดจะต่าง): ตรง prompt 37.6/37.4/37.0/36.1 พอกัน (ช่วง noise) · CLIP ภาพ-ภาพ 87.5/85.3/88.0/87.1 ก็อยู่ในช่วง noise ±3 ทั้งที่ตาเห็นชัดว่า C/D เป็นกาน้ำชาดินเผาของร้าน → ตามกติกา ตัววัดนี้ตอบ "ใบนี้ไหม" ไม่ได้ (ทุกกาน้ำชา 85+) ตาเป็นผู้ตัดสิน · diversity: A 36.7 สูงสุด ลดเมื่อใส่ ControlNet (31.2) และลดเมื่อใส่ LoRA (34.9) ด้วย — การบังคับทุกแบบลดความหลากหลาย · บทเรียน: ตัวเลขแต่ละตัวตอบคำถามเฉพาะของมัน (ตรงคำ/หลากหลาย ใช้ตัวเลข · ใบนี้ไหม ใช้ตา หรือ DINO — โบนัส)

In [ ]:
# @snip:final
cn_scr = ControlNetModel.from_pretrained("lllyasviel/sd-controlnet-scribble", torch_dtype=DT)
cnp = StableDiffusionControlNetPipeline.from_pipe(pipe, controlnet=cn_scr).to(DEV); cnp.set_progress_bar_config(disable=True)
P_FINAL = "{} on a wooden table in a cozy cafe, empty wall on the right, product photography, soft morning light"
SEEDS = [42, 7, 123, 2024]
def gen_set(use_lora, use_cn):
    scale = 1.0 if use_lora else 0.0; prompt = P_FINAL.format(TRIGGER if use_lora else "a teapot"); out = []
    for s in SEEDS:
        if use_cn:
            out.append(cnp(prompt, image=layout, controlnet_conditioning_scale=1.0, negative_prompt=NEG,   # โครงภาพจาก 3.4 (14 px) scale เดียวกับ 3.4
                           num_inference_steps=STEPS, generator=gen(s), cross_attention_kwargs={"scale": scale}).images[0])
        else:
            out.append(pipe(prompt, negative_prompt=NEG, num_inference_steps=STEPS, generator=gen(s),
                            cross_attention_kwargs={"scale": scale}).images[0])
    return out
sets = {"A prompt": gen_set(False, False), "B +ControlNet": gen_set(False, True),
        "C +LoRA": gen_set(True, False), "D LoRA+ControlNet": gen_set(True, True)}
rows = []
for name, ims in sets.items():
    rows.append(dict(แนวทาง=name, CLIP_ภาพ_ข้อความ=clip_score(ims, P_FINAL.format("a teapot")).mean(),
                     CLIP_ภาพ_ภาพ=clip_sim_to_refs(ims).mean(), diversity=diversity(ims, 50)))
import pandas as pd; df = pd.DataFrame(rows).round(1); print(df.to_string(index=False))
df.to_json("week5_results.json", orient="records", force_ascii=False, indent=1)
show_grid([im for ims in sets.values() for im in ims], [f"{n} · seed {s}" for n in sets for s in SEEDS], cols=4, save="lab5_final")

### สรุปสิ่งที่ได้วันนี้
| ความต้องการของร้าน | ทำได้ด้วย | แล็บ |
|---|---|---|
| ภาพใหญ่ชัด | latent diffusion (VAE + U-Net ขนาดจริง) | 1 |
| สั่งเป็นประโยค | CLIP + cross-attention + prompt ที่เขียนแบบ caption | 1–2 |
| จัดวางตามเลย์เอาต์ | img2img / inpainting / ControlNet | 3 |
| เป็นกาน้ำชาของร้านจริง | DreamBooth-LoRA | 4 |

**การบ้าน** "Prompt, ControlNet หรือ LoRA? ตัดสินด้วยตาและตัวเลข" บนวัตถุของคุณเอง (ภาพ 5–8 ใบจากมือถือ) ใช้ `HW5_Starter.ipynb` — memo ต้องมีภาพประกอบคู่ตาราง